# Comparative Analysis of RNN, GRU, LSTM, and Word2Vec for Sentiment Classification
**Dataset:** Toxic Tweet Dataset | **Task:** Binary Sentiment Classification (Toxic vs Non-Toxic)

## Cell 1 — Install & Import Libraries

In [ ]:
# Cell 1: Install & Import Libraries
!pip install gensim numpy pandas scikit-learn tensorflow matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, GRU, LSTM,
    Dense, BatchNormalization, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import gensim
from gensim.models import Word2Vec

print('All libraries imported successfully.')

## Cell 2 — Load Dataset

In [ ]:
# Cell 2: Load Dataset
# Dataset: Toxic Tweet Dataset (label 0 = non-toxic, label 1 = toxic/sexist/racist)
# Source: https://www.kaggle.com/datasets/vikasukani/hate-speech-and-offensive-content-identification

df = pd.read_csv('twitter_data.csv')   # adjust filename as needed
print('Shape:', df.shape)
print(df.head())
print('\nClass distribution:')
print(df['label'].value_counts())

## Cell 3 — Dataset Preparation (Undersampling)

In [ ]:
# Cell 3: Dataset Preparation — Undersampling to balance classes
# Original: ~29,720 non-toxic | ~2,242 toxic

toxic     = df[df['label'] == 1]
non_toxic = df[df['label'] == 0]

# Undersample majority class to match minority class size
non_toxic_sampled = non_toxic.sample(n=len(toxic), random_state=42)

balanced_df = pd.concat([toxic, non_toxic_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Balanced dataset shape:', balanced_df.shape)
print('Class distribution after undersampling:')
print(balanced_df['label'].value_counts())

## Cell 4 — Text Preprocessing

In [ ]:
# Cell 4: Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'@\w+', '', text)                   # remove mentions
    text = re.sub(r'[^a-zA-Z\s]', '', text)            # remove non-alpha
    text = re.sub(r'\s+', ' ', text).strip()           # normalize whitespace
    return text

balanced_df['clean_text'] = balanced_df['tweet'].apply(clean_text)

texts  = balanced_df['clean_text'].tolist()
labels = balanced_df['label'].values

print('Sample cleaned tweet:')
print(texts[0])

## Cell 5 — Tokenization & Sequence Padding

In [ ]:
# Cell 5: Tokenization & Padding
MAX_LEN    = 42      # fixed sequence length
VOCAB_SIZE = None    # determined after fitting tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

VOCAB_SIZE = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {VOCAB_SIZE}')

sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = labels

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## Cell 6 — Train/Test Split

In [ ]:
# Cell 6: Train/Test Split (80-20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]}')
print(f'Test size  : {X_test.shape[0]}')

## Cell 7 — Word2Vec Embedding Matrix

In [ ]:
# Cell 7: Train Word2Vec & Build Embedding Matrix
EMBED_DIM = 128

# Tokenize sentences into word lists for Word2Vec training
w2v_sentences = [text.split() for text in texts]

w2v_model = Word2Vec(
    sentences=w2v_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    sg=1,              # skip-gram
    workers=4,
    epochs=10,
    seed=42
)

# Build embedding matrix aligned with Keras tokenizer index
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
for word, idx in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f'Embedding matrix shape: {embedding_matrix.shape}')
print(f'Non-zero vectors      : {np.count_nonzero(embedding_matrix.sum(axis=1))}')

## Cell 8 — Model Builder Function

In [ ]:
# Cell 8: Generic Model Builder
# All models share identical architecture except the recurrent layer & embedding init

def build_model(model_type='LSTM', embedding_matrix=None):
    """
    model_type       : 'RNN' | 'GRU' | 'LSTM' | 'Word2Vec'
    embedding_matrix : numpy array for Word2Vec, else None (trainable embedding)
    """
    model = Sequential(name=model_type)

    # --- Embedding Layer ---
    if embedding_matrix is not None:
        model.add(Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBED_DIM,
            weights=[embedding_matrix],
            input_length=MAX_LEN,
            trainable=False,          # freeze pretrained vectors
            name='embedding_w2v'
        ))
    else:
        model.add(Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBED_DIM,
            input_length=MAX_LEN,
            trainable=True,
            name='embedding_trainable'
        ))

    # --- Recurrent Layer (128 units) ---
    if model_type == 'RNN':
        model.add(SimpleRNN(128, name='simple_rnn'))
    elif model_type == 'GRU':
        model.add(GRU(128, name='gru'))
    elif model_type in ('LSTM', 'Word2Vec'):
        model.add(LSTM(128, name='lstm'))

    # --- Dense Layers (identical across all models) ---
    model.add(Dense(256, activation='relu', name='dense_256'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu', name='dense_128'))
    model.add(BatchNormalization())

    # --- Output Layer ---
    model.add(Dense(1, activation='sigmoid', name='output'))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick sanity check
build_model('LSTM').summary()

## Cell 9 — Train All Models

In [ ]:
# Cell 9: Train All Four Models
EPOCHS     = 10
BATCH_SIZE = 64
VAL_SPLIT  = 0.1

histories = {}
trained_models = {}

configs = [
    ('RNN',      None),
    ('GRU',      None),
    ('LSTM',     None),
    ('Word2Vec', embedding_matrix),
]

for name, emb_matrix in configs:
    print(f'\n{'='*50}')
    print(f'Training: {name}')
    print('='*50)

    model = build_model(model_type=name, embedding_matrix=emb_matrix)

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VAL_SPLIT,
        verbose=1
    )

    histories[name]      = history
    trained_models[name] = model

print('\nAll models trained.')

## Cell 10 — Evaluate All Models

In [ ]:
# Cell 10: Evaluate All Models on Test Set
results = {}

for name, model in trained_models.items():
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred      = (y_pred_prob >= 0.5).astype(int).flatten()

    report = classification_report(y_test, y_pred, target_names=['Non-Toxic', 'Toxic'], output_dict=True)
    acc    = accuracy_score(y_test, y_pred)

    results[name] = {
        'accuracy'  : round(acc, 4),
        'precision' : round(report['weighted avg']['precision'], 4),
        'recall'    : round(report['weighted avg']['recall'], 4),
        'f1'        : round(report['weighted avg']['f1-score'], 4),
    }

    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Non-Toxic', 'Toxic']))

# Summary table
results_df = pd.DataFrame(results).T.rename(columns={
    'accuracy': 'ACC', 'precision': 'PREC', 'recall': 'REC', 'f1': 'F1'
})
print('\n===== Summary Table =====')
print(results_df)

## Cell 11 — Plot Training Accuracy & Loss Curves

In [ ]:
# Cell 11: Plot Accuracy & Loss Curves for All Models
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
colors = {'RNN': '#E63946', 'GRU': '#2A9D8F', 'LSTM': '#E9C46A', 'Word2Vec': '#457B9D'}

for col, (name, history) in enumerate(histories.items()):
    c = colors[name]

    # Accuracy
    axes[0, col].plot(history.history['accuracy'],     color=c, label='Train')
    axes[0, col].plot(history.history['val_accuracy'], color=c, linestyle='--', label='Val')
    axes[0, col].set_title(f'{name} — Accuracy')
    axes[0, col].set_xlabel('Epoch')
    axes[0, col].set_ylabel('Accuracy')
    axes[0, col].legend()
    axes[0, col].grid(alpha=0.3)

    # Loss
    axes[1, col].plot(history.history['loss'],     color=c, label='Train')
    axes[1, col].plot(history.history['val_loss'], color=c, linestyle='--', label='Val')
    axes[1, col].set_title(f'{name} — Loss')
    axes[1, col].set_xlabel('Epoch')
    axes[1, col].set_ylabel('Loss')
    axes[1, col].legend()
    axes[1, col].grid(alpha=0.3)

plt.suptitle('Training & Validation Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 12 — Bar Chart: Model Comparison

In [ ]:
# Cell 12: Bar Chart — Comparative Metrics
metrics = ['ACC', 'PREC', 'REC', 'F1']
x       = np.arange(len(metrics))
width   = 0.18
colors  = ['#E63946', '#2A9D8F', '#E9C46A', '#457B9D']

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, row) in enumerate(results_df.iterrows()):
    bars = ax.bar(x + i * width, row[metrics].values, width, label=name, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0.75, 0.92)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: RNN vs GRU vs LSTM vs Word2Vec', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 13 — Inference on New Text

In [ ]:
# Cell 13: Inference on Custom Input
def predict_sentiment(text, model, tokenizer, max_len=42):
    cleaned  = clean_text(text)
    seq      = tokenizer.texts_to_sequences([cleaned])
    padded   = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    prob     = model.predict(padded, verbose=0)[0][0]
    label    = 'TOXIC' if prob >= 0.5 else 'NON-TOXIC'
    return label, round(float(prob), 4)

sample_texts = [
    "I hate all people like you, you're disgusting!",
    "What a beautiful day, hope you have a great time!",
    "Women should stay in the kitchen and not speak.",
]

print(f'{"Text":<55} {"RNN":<12} {"GRU":<12} {"LSTM":<12} {"Word2Vec"}')
print('-' * 100)
for text in sample_texts:
    preds = [predict_sentiment(text, trained_models[m], tokenizer) for m in ['RNN','GRU','LSTM','Word2Vec']]
    row   = '  '.join([f'{label}({prob})' for label, prob in preds])
    print(f'{text[:52]:<55} {row}')